# 04 - ElasticNet y Comparativa Integral de Modelos Lineales

## Curso de Machine Learning: Modelos de Regresion

En este cuaderno estudiaremos **ElasticNet**, un estimador propuesto por Zou y Hastie (2005) que combina de manera convexa las penalizaciones $L_1$ (Lasso) y $L_2$ (Ridge). Aprenderemos por que ElasticNet supera las limitaciones de Lasso en presencia de grupos de variables fuertemente correlacionadas o cuando el numero de caracteristicas es superior al de observaciones ($p > n$). Finalmente, realizaremos un benchmark integral comparando OLS, Ridge, Lasso y ElasticNet sobre el dataset de diabetes.

---

### Contenidos
1. Motivacion y Limitaciones de Lasso Puro
2. Formulacion Matematica de ElasticNet
3. Los Hiperparametros $\alpha$ y `l1_ratio`
4. Optimizacion con `ElasticNetCV`
5. Benchmark Global: OLS vs. Ridge vs. Lasso vs. ElasticNet
6. Analisis Comparativo de Coeficientes y Seleccion de Caracteristicas
7. Evaluacion Visual: Predicciones vs. Valores Reales y Residuos
8. Guia Practica de Decision: ¿Que Modelo Lineal Elegir en la Practica?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Motivacion y Limitaciones de Lasso Puro

Aunque Lasso es una herramienta potente para la seleccion de variables, presenta tres debilidades teoricas y operativas conocidas:
1. **Seleccion arbitraria en grupos correlacionados**: Si un conjunto de variables exhibe alta colinealidad mutua (como los perfiles de suero `s1`, `s2`, `s4`), Lasso tiende a seleccionar aleatoriamente una de ellas y enviar las demas a cero, sin importar que todas aporten senal biologica o fisica relevante.
2. **Cota de dimensionalidad ($p > n$)**: Si el numero de predictores $p$ supera el numero de observaciones de entrenamiento $n$, Lasso puede seleccionar como maximo $n$ variables antes de saturarse.
3. **Penalizacion excesiva de variables dominantes**: La penalizacion $L_1$ constante puede sesgar fuertemente los coeficientes de senales genuinas de gran magnitud.

**Solucion**: Incorporar una componente cuadratica $L_2$ (estilo Ridge) que conserve el efecto de agrupamiento (grouping effect) y mantenga la capacidad de anular variables irrelevantes mediante la componente $L_1$.


## 2. Formulacion Matematica de ElasticNet

La funcion objetivo de ElasticNet combina las normas $L_1$ y $L_2^2$:

$$J_{\text{ElasticNet}}(\beta) = \frac{1}{2n} \|y - X\beta\|_2^2 + \alpha \cdot \rho \|\beta\|_1 + \frac{\alpha (1 - \rho)}{2} \|\beta\|_2^2$$

Donde:
- $\alpha \ge 0$: Fuerza global de la penalizacion.
- $\rho \in [0, 1]$ (denominado `l1_ratio` en Scikit-Learn): Proporcion de la penalizacion correspondiente a la norma $L_1$:
  - Si $\rho = 1$: El modelo es estrictamente **Lasso**.
  - Si $\rho = 0$: El modelo es estrictamente **Ridge**.
  - Si $0 < \rho < 1$: El modelo es un compromiso equilibrado entre ambas tecnicas.


## 3. Carga y Particion del Dataset Diabetes

Utilizamos la misma particion de datos para garantizar una comparativa justa y reproducible entre todos los algoritmos.


In [ ]:
diabetes = load_diabetes(scaled=False, as_frame=True)
X = diabetes.frame.drop(columns=['target'])
y = diabetes.frame['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Dataset de diabetes cargado con exito.")
print(f"Muestras de entrenamiento: {X_train.shape[0]} | Muestras de prueba: {X_test.shape[0]}")


## 4. Optimizacion de Hiperparametros con `ElasticNetCV`

Scikit-Learn ofrece `ElasticNetCV`, el cual realiza una busqueda por validacion cruzada sobre una cuadricula bidimensional combinando valores de `l1_ratio` y una secuencia automatica de $\alpha$.


In [ ]:
# Definicion de una malla de l1_ratio
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]

pipeline_enet_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('enet_cv', ElasticNetCV(l1_ratio=l1_ratios, cv=5, random_state=42, max_iter=10000))
])

pipeline_enet_cv.fit(X_train, y_train)

best_enet = pipeline_enet_cv.named_steps['enet_cv']
print(f"Mejor alpha seleccionado:    {best_enet.alpha_:.4f}")
print(f"Mejor l1_ratio seleccionado: {best_enet.l1_ratio_:.2f}")


## 5. Benchmark Global: OLS vs. Ridge vs. Lasso vs. ElasticNet

Entrenamos los cuatro modelos lineales bajo identicas condiciones de preprocesamiento (usando `StandardScaler` en `Pipeline`) y comparamos su rendimiento predictivo tanto en entrenamiento como en prueba.


In [ ]:
# 1. Ajuste de todos los modelos optimizados
modelos = {
    'OLS (Sin Regularizar)': Pipeline([
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ]),
    'Ridge (L2)': Pipeline([
        ('scaler', StandardScaler()),
        ('reg', RidgeCV(cv=5))
    ]),
    'Lasso (L1)': Pipeline([
        ('scaler', StandardScaler()),
        ('reg', LassoCV(cv=5, random_state=42, max_iter=10000))
    ]),
    'ElasticNet (L1 + L2)': Pipeline([
        ('scaler', StandardScaler()),
        ('reg', ElasticNetCV(l1_ratio=l1_ratios, cv=5, random_state=42, max_iter=10000))
    ])
}

resultados = []

for nombre, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    
    # Predicciones
    y_pred_tr = pipe.predict(X_train)
    y_pred_te = pipe.predict(X_test)
    
    # Metricas
    mse_tr = mean_squared_error(y_train, y_pred_tr)
    mse_te = mean_squared_error(y_test, y_pred_te)
    rmse_te = np.sqrt(mse_te)
    mae_te = mean_absolute_error(y_test, y_pred_te)
    r2_tr = r2_score(y_train, y_pred_tr)
    r2_te = r2_score(y_test, y_pred_te)
    
    # Extraccion de coeficientes
    reg_step = pipe.named_steps['reg']
    coefs = reg_step.coef_
    n_activos = np.sum(coefs != 0)
    
    resultados.append({
        'Modelo': nombre,
        'MSE_Train': mse_tr,
        'MSE_Test': mse_te,
        'RMSE_Test': rmse_te,
        'MAE_Test': mae_te,
        'R2_Train': r2_tr,
        'R2_Test': r2_te,
        'Vars_Activas': f"{n_activos}/{len(X.columns)}"
    })

df_benchmark = pd.DataFrame(resultados)
print("Tabla Comparativa Integral de Rendimiento:")
print(df_benchmark.round(4).to_string(index=False))


## 6. Analisis Comparativo de Coeficientes y Seleccion de Caracteristicas

Comparamos los valores numericos de los coeficientes normalizados para cada uno de los 4 estimadores.


In [ ]:
# Construccion del DataFrame de coeficientes
dict_coefs = {'Variable': X.columns}
for nombre, pipe in modelos.items():
    dict_coefs[nombre] = pipe.named_steps['reg'].coef_

df_coefs_comp = pd.DataFrame(dict_coefs)
print("Coeficientes estandarizados por modelo:")
print(df_coefs_comp.round(3).to_string(index=False))

# Grafico de barras agrupadas
fig, ax = plt.subplots(figsize=(14, 6))

indices = np.arange(len(X.columns))
ancho = 0.20
colores = ['#d95f02', '#2b5c8f', '#1b9e77', '#7570b3']

for i, (nombre, color) in enumerate(zip(modelos.keys(), colores)):
    ax.bar(indices + i*ancho - 1.5*ancho, df_coefs_comp[nombre], width=ancho, label=nombre, color=color)

ax.set_xticks(indices)
ax.set_xticklabels(X.columns, fontsize=11)
ax.set_ylabel('Magnitud del Coeficiente', fontsize=11, fontweight='bold')
ax.set_title('Comparativa de Coeficientes: OLS vs. Ridge vs. Lasso vs. ElasticNet', fontsize=13, fontweight='bold')
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


## 7. Evaluacion Visual: Predicciones vs. Valores Reales y Residuos

Analizamos graficamente la correlacion entre las observaciones reales $y$ y las predicciones $\hat{y}$ del modelo ElasticNet en el conjunto de prueba.


In [ ]:
y_pred_enet = modelos['ElasticNet (L1 + L2)'].predict(X_test)
residuos_enet = y_test - y_pred_enet

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Panel 1: Predicciones vs Valores Reales
axes[0].scatter(y_test, y_pred_enet, color='#7570b3', alpha=0.8, edgecolor='k', s=50)
min_val = min(y_test.min(), y_pred_enet.min())
max_val = max(y_test.max(), y_pred_enet.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prediccion Perfecta (y = y_hat)')
axes[0].set_xlabel('Valor Real de Progresion (y)', fontsize=11)
axes[0].set_ylabel('Prediccion ElasticNet ($\\hat{y}$)', fontsize=11)
axes[0].set_title('Valores Reales vs. Predicciones (ElasticNet)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Panel 2: Residuos vs Predicciones
axes[1].scatter(y_pred_enet, residuos_enet, color='#7570b3', alpha=0.8, edgecolor='k', s=50)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediccion ElasticNet ($\\hat{y}$)', fontsize=11)
axes[1].set_ylabel('Residuo ($y - \\hat{y}$)', fontsize=11)
axes[1].set_title('Residuos vs. Predicciones (ElasticNet)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Guia Practica de Decision: ¿Que Modelo Lineal Elegir?

Para finalizar este modulo, resumimos los criterios de seleccion de modelos lineales en proyectos de ciencia de datos:

| Situacion / Caracteristica del Problema | Modelo Recomendado | Justificacion |
| :--- | :--- | :--- |
| Pocas variables ($p \ll n$), sin multicolinealidad, interpretabilidad clasica requerida | **OLS (LinearRegression)** | Estimador insesgado de varianza minima (Gauss-Markov). |
| Muchas variables, alta multicolinealidad entre variables, todas se consideran relevantes | **Ridge ($L_2$)** | Estabiliza la inversion matricial reduciendo la varianza sin descartar variables. |
| Gran numero de caracteristicas, sospecha de que muchas son ruido o redundantes | **Lasso ($L_1$)** | Anula coeficientes de forma exacta, produciendo modelos parsimoniosos. |
| Variables fuertemente correlacionadas en grupos, o alta dimensionalidad ($p > n$) | **ElasticNet ($L_1 + L_2$)** | Mantiene seleccion de caracteristicas pero preserva grupos correlacionados. |
